In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Methodology.EvaluationELM import EvaluationELM
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_cleaned_data_path()
gallstone_dataset.normal_data_split()

features_size = gallstone_dataset.x_train.shape[1]

In [3]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x_train = gallstone_dataset.x_train,
    y_train = gallstone_dataset.y_train,
    activation_function     = GlobalSetting.sigmoid ,
    elm_init_seed_range     = GlobalSetting.elm_initial_state_range ,
    k_fold = 5
)

In [4]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(
    GlobalSetting.hidden_size_explore_range
)
GlobalSetting.save_dataframe_to_record(hidden_size_record,'hidden_size_record.csv')
hidden_node_list = EvaluationELM.extract_top_results(hidden_size_result)
best_hidden_size = hidden_node_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\hidden_size_record.csv


In [5]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size     = best_hidden_size['Hidden_Nodes'],
    lambda_range    = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(lambda_record,'lambda_record.csv')
lambda_value_list = EvaluationELM.extract_top_results(lambda_result)
best_lambda_value = lambda_value_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\lambda_record.csv


In [6]:
size_lambda_record , size_lambda_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = hidden_node_list['Hidden_Nodes'],
    lambda_range        = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(size_lambda_record,'size_lambda_record.csv')
size_lambda_list = EvaluationELM.extract_top_results(size_lambda_result)
best_size_lambda = size_lambda_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\size_lambda_record.csv


In [7]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = GlobalSetting.hidden_size_explore_range,
    lambda_range        = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(combination_record,'combination_record')
combination_result_list = EvaluationELM.extract_top_results(combination_result)
best_combination_result = combination_result_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\combination_record.csv


In [8]:
hidden_size_smaller = range(11, round(best_hidden_size['Hidden_Nodes'] * 0.8))
optimization_record , optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range = hidden_size_smaller,
    lambda_range = GlobalSetting.lambda_search_range
)
GlobalSetting.save_dataframe_to_record(optimization_record,'optimization_record.csv')
optimization_result_list = EvaluationELM.extract_top_results(optimization_result)
best_optimization_result = optimization_result_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\optimization_record.csv


In [9]:
model_configs_payload = [
    {
        "Model_Types" : "Best_Hidden_Nodes",
        "Hidden_Nodes": int(best_hidden_size['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_hidden_size['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Lambda",
        "Hidden_Nodes": int(best_lambda_value['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_lambda_value['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Size_and_Lambda",
        "Hidden_Nodes": int(best_size_lambda['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_size_lambda['Lambda_Value'])
    },
    {
        "Model_Types" : "Grid_Combination",
        "Hidden_Nodes": int(best_combination_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_combination_result['Lambda_Value'])
    },
    {
        "Model_Types" : "Grid_Optimization",
        "Hidden_Nodes": int(best_optimization_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_optimization_result['Lambda_Value'])
    }
]
GlobalSetting.upsert_model_configs(model_configs_payload)

In [10]:
combined_df = pd.concat([best_hidden_size, best_lambda_value, best_size_lambda,best_combination_result, best_optimization_result], axis=1)
combined_df

,28,21,134,2459,1087
Hidden_Nodes,39,39,43,98,30
Activation,sigmoid,sigmoid,sigmoid,sigmoid,sigmoid
Lambda_Value,0.0,0.0625,0.125,0.25,0.0625
avg_Accuracy_Seed_Mean,0.779537,0.78435,0.792667,0.808559,0.772434
avg_Accuracy_Seed_Std,0.016371,0.013136,0.016237,0.010899,0.020775
avg_Accuracy_Seed_SEM,0.002989,0.002398,0.002964,0.00199,0.003793
avg_Accuracy_Seed_Min,0.740314,0.76,0.760314,0.775765,0.732471
avg_Accuracy_Seed_Max,0.819137,0.807373,0.831137,0.823216,0.807216
avg_Precision_Seed_Mean,0.822345,0.831659,0.838108,0.862047,0.814735
avg_Precision_Seed_Std,0.022831,0.017767,0.022208,0.015887,0.026268
